In [16]:
# Raw dataset filtering for the RAG pipeline
#
# Obiettivo:
# - caricare il dataset raw di regole trigger-action;
# - normalizzare i nomi dei canali;
# - costruire e completare la channel taxonomy;
# - filtrare le regole rilevanti per il dominio smart home;
# - esportare un subset iniziale da usare nelle fasi successive del progetto RAG.

In [17]:
import pandas as pd
import re
from pathlib import Path

In [ ]:
# Path principali del progetto
DATA_DIR = Path("../../data")
RAW_DATA_PATH = DATA_DIR / "dataset_raw.csv"
CHANNEL_TAXONOMY_PATH = DATA_DIR / "channel_taxonomy.csv"
CHANNEL_TAXONOMY_FINAL_PATH = DATA_DIR / "channel_taxonomy_final.csv"
CANDIDATES_PATH = DATA_DIR / "dataset_candidates_rag.csv"
REVIEW_PATH = DATA_DIR / "dataset_review_rag.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

In [19]:
def normalize_channel_name(name: str) -> str:
    """
    Normalizza i nomi canale per gestire problemi di encoding e varianti
    ortografiche presenti nel dataset raw e nel taxonomy CSV.

    L'obiettivo è portare i nomi a una forma canonica e stabile, così da
    rendere affidabile il merge tra dataset_raw e channel_taxonomy.
    """
    if not isinstance(name, str) or not name:
        return name

    result = name.strip()

    # Canonicalizzazione di varianti note
    result = re.sub(r"tado.{0,8}Heating", "tado Heating", result)
    result = re.sub(r"tado.{0,8}Air Conditioning", "tado Air Conditioning", result)
    result = re.sub(r"Lutron Cas.{0,8}ta Wireless", "Lutron Caseta Wireless", result)

    def _hive_heat(match):
        return "Hive Active Heating [beta]" if "[beta]" in match.group(0) else "Hive Active Heating"

    result = re.sub(r"Hive Active Heating.{0,15}?(\[beta\])?$", _hive_heat, result)
    result = re.sub(r"Hive Active Light.{0,10}$", "Hive Active Light", result)
    result = re.sub(r"GE Appliances GeoSpring.{0,10}$", "GE Appliances GeoSpring", result)
    result = re.sub(r"Donation Manager RedCloud.{0,10}$", "Donation Manager RedCloud", result)

    # Rimozione di eventuali caratteri non ASCII residui
    result = re.sub(r"[^\x00-\x7F]+", "", result).strip()

    return result

In [20]:
# Caricamento del dataset raw
df = pd.read_csv(RAW_DATA_PATH, sep=";", encoding="latin1")

print("Shape:", df.shape)
display(df.head())

Shape: (79214, 6)


,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc
0,New Popular photo,500px,Update device wallpaper,Android Device,"If there is a new photo in 500px ,then but it ...",You can change the category
1,New Popular photo,500px,Update device wallpaper,Android Device,Any 500px Popular photo on Wallpaper,Turn notifications off
2,New Popular photo,500px,Update device wallpaper,Android Device,Popular Images from 500px on Device,Just click and enable to have the 500 Px popul...
3,New Popular photo,500px,Update device wallpaper,Android Device,any popular photo on 500px on my mobile,allows to have series of wonderful pictures as...
4,New photo from search,500px,Update device wallpaper,Android Device,New york,If new photo by anyone matching search new yor...


In [21]:
# Normalizzazione dei canali di trigger e action
df["triggerChannelTitle"] = df["triggerChannelTitle"].apply(normalize_channel_name)
df["actionChannelTitle"] = df["actionChannelTitle"].apply(normalize_channel_name)

In [22]:
# Ispezione rapida del dataset
print("Colonne:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nTop trigger channels:")
display(df["triggerChannelTitle"].value_counts().head(20))

print("\nTop action channels:")
display(df["actionChannelTitle"].value_counts().head(20))

Colonne:
['triggerTitle', 'triggerChannelTitle', 'actionTitle', 'actionChannelTitle', 'title', 'desc']

Missing values:


desc                   96
title                  13
triggerChannelTitle     0
triggerTitle            0
actionChannelTitle      0
actionTitle             0
dtype: int64


Top trigger channels:


triggerChannelTitle
RSS Feed               31040
Weather Underground     3956
Date & Time             3330
Twitter                 3302
YouTube                 2393
Gmail                   2378
Instagram               2086
reddit                  1625
WordPress               1557
Location                1411
Classifieds             1314
Facebook                1258
Facebook Pages          1239
Blogger                 1058
Amazon Alexa             935
Pocket                   867
Google Calendar          793
Dropbox                  716
ESPN                     695
iOS Photos               634
Name: count, dtype: int64


Top action channels:


actionChannelTitle
Twitter           9092
Google Drive      5270
Notifications     4201
Email             3865
Evernote          3623
Tumblr            3516
SMS               2880
Gmail             2826
Pocket            2675
Facebook Pages    2597
Delicious         2562
Buffer            2453
Facebook          2236
Instapaper        1916
Dropbox           1651
OneNote           1611
Diigo             1434
Android Device    1343
WordPress         1320
bitly             1255
Name: count, dtype: int64

In [23]:
# Costruzione dell'elenco dei canali unici considerando sia trigger che action
channels = pd.concat(
    [df["triggerChannelTitle"], df["actionChannelTitle"]],
    ignore_index=True
)

channels = channels.fillna("").astype(str).str.strip()
channels = channels[channels != ""]

unique_channels = sorted(channels.unique())
print("Numero canali unici:", len(unique_channels))
print(unique_channels[:50])

Numero canali unici: 429
['500px', 'ARTIK Cloud', 'AT&T M2X', 'Adafruit', 'Airtable', 'Almond', 'Amazon Alexa', 'Amazon Alexa (US only)', 'Amazon Cloud Drive', 'Ambi Climate', 'Android Battery', 'Android Device', 'Android Location', 'Android Phone Call', 'Android Photos', 'Android SMS', 'Android Wear', 'AppZapp', 'Arlo', 'Arlo QA (Staging)', 'Asana', 'August', 'Automatic Classic', 'Automatic Pro', 'Awair', 'BMW Labs', "Bang & Olufsen's BeoLink Gateway", 'Beeminder', 'Beseye', 'Beseye_dev', 'Best Buy', 'Bhome', 'Blogger', 'BloomSky Weather', 'Box', 'Boxcar 2', 'Boxoh Package Tracking', 'Bttn', 'Buffer', 'Button widget', 'CONNECTED WATCH', 'Caleo', 'Caleo (Staging)', 'Camera widget', 'Camio', 'Chain', 'Cisco Spark', 'Classifieds', 'Code School', 'Cogitai']


In [24]:
# Conteggio di frequenza per ciascun canale
channel_counts = channels.value_counts()

display(channel_counts.head(100))

RSS Feed                   31040
Twitter                    12394
Google Drive                5270
Gmail                       5204
Email                       4401
                           ...  
Netatmo Weather Station      106
Salesforce                   106
Ubi                          103
Nest Protect                 102
Sina Weibo                    96
Name: count, Length: 100, dtype: int64

In [25]:
# Creazione o aggiornamento del taxonomy dei canali
channels_df = channel_counts.reset_index()
channels_df.columns = ["channel_name", "frequency"]

if CHANNEL_TAXONOMY_PATH.exists():
    taxonomy_df = pd.read_csv(CHANNEL_TAXONOMY_PATH, sep=";", encoding="utf-8")
    channels_df = channels_df.merge(
        taxonomy_df[["channel_name", "class"]],
        on="channel_name",
        how="left"
    )
else:
    channels_df["class"] = ""

channels_df["class"] = channels_df["class"].fillna("")

channels_df.to_csv(CHANNEL_TAXONOMY_PATH, sep=";", index=False, encoding="utf-8")
print(f"Salvato taxonomy intermedio in: {CHANNEL_TAXONOMY_PATH}")

Salvato taxonomy intermedio in: ..\data\channel_taxonomy.csv


In [26]:
# Pulizia finale della taxonomy usando channel_class_mapping.csv come fonte delle classi
CHANNEL_CLASS_MAPPING_PATH = DATA_DIR / "channel_class_mapping.csv"

tax = pd.read_csv(CHANNEL_TAXONOMY_PATH, sep=";", encoding="utf-8")
mapping = pd.read_csv(CHANNEL_CLASS_MAPPING_PATH, sep=";", encoding="utf-8")

# Teniamo dal taxonomy solo le colonne strutturali
tax = tax[["channel_name", "frequency"]].copy()

# Teniamo dal mapping solo la mappa nome -> classe
mapping = mapping[["channel_name", "class"]].copy()

# Normalizzazione nomi canale
tax["channel_name"] = tax["channel_name"].astype(str).str.strip().apply(normalize_channel_name)
mapping["channel_name"] = mapping["channel_name"].astype(str).str.strip().apply(normalize_channel_name)

# Pulizia colonna class nel mapping
mapping["class"] = mapping["class"].astype(str).str.strip()

# Rimuove duplicati nel mapping dopo normalizzazione
mapping = mapping.drop_duplicates(subset=["channel_name"], keep="first")

# Merge pulito: tax + classi dal mapping
tax = tax.merge(mapping, on="channel_name", how="left")

# Forza class a stringa
tax["class"] = tax["class"].fillna("").astype(str).str.strip()

# Fix manuali per varianti o canali non coperti bene
manual_fixes = {
    "GE Appliances GeoSpring": "smart_home",
    "tado Heating": "smart_home",
    "tado Air Conditioning": "smart_home",
    "Hive Active Heating": "smart_home",
    "Hive Active Light": "smart_home",
    "Lutron Caseta Wireless": "smart_home",
    "Donation Manager RedCloud": "general_digital"
}

for channel, cls in manual_fixes.items():
    tax.loc[tax["channel_name"] == channel, "class"] = cls

# Gestione di una variante nota rimasta nel taxonomy
tax.loc[tax["channel_name"] == "Lutron Casta Wireless", "channel_name"] = "Lutron Caseta Wireless"
tax.loc[tax["channel_name"] == "Lutron Caseta Wireless", "class"] = "smart_home"

# Se dopo la rinomina si generano duplicati, somma le frequenze e tieni una sola riga
tax = (
    tax.groupby(["channel_name", "class"], as_index=False)["frequency"]
       .sum()
       .sort_values("frequency", ascending=False)
)

missing = tax[tax["class"] == ""]
print("Canali senza classe:", len(missing))
if len(missing) > 0:
    display(missing.sort_values("frequency", ascending=False))

print("\nDistribuzione classi:")
print(tax["class"].value_counts(dropna=False).to_string())

tax.to_csv(CHANNEL_TAXONOMY_FINAL_PATH, sep=";", index=False, encoding="utf-8-sig")
print(f"\nSalvato taxonomy finale in: {CHANNEL_TAXONOMY_FINAL_PATH}")

Canali senza classe: 0

Distribuzione classi:
class
smart_home         199
general_digital    171
unknown             32
context             26

Salvato taxonomy finale in: ..\data\channel_taxonomy_final.csv


In [27]:
# Caricamento della taxonomy finale e preparazione al merge
tax = pd.read_csv(CHANNEL_TAXONOMY_FINAL_PATH, sep=";", encoding="utf-8-sig")

tax["channel_name"] = tax["channel_name"].apply(normalize_channel_name)

trigger_map = tax[["channel_name", "class"]].rename(columns={
    "channel_name": "triggerChannelTitle",
    "class": "trigger_class"
})

action_map = tax[["channel_name", "class"]].rename(columns={
    "channel_name": "actionChannelTitle",
    "class": "action_class"
})

In [28]:
# Merge delle classi sul dataset raw
df = df.merge(trigger_map, on="triggerChannelTitle", how="left")
df = df.merge(action_map, on="actionChannelTitle", how="left")

df.head()

,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc,trigger_class,action_class
0,New Popular photo,500px,Update device wallpaper,Android Device,"If there is a new photo in 500px ,then but it ...",You can change the category,general_digital,general_digital
1,New Popular photo,500px,Update device wallpaper,Android Device,Any 500px Popular photo on Wallpaper,Turn notifications off,general_digital,general_digital
2,New Popular photo,500px,Update device wallpaper,Android Device,Popular Images from 500px on Device,Just click and enable to have the 500 Px popul...,general_digital,general_digital
3,New Popular photo,500px,Update device wallpaper,Android Device,any popular photo on 500px on my mobile,allows to have series of wonderful pictures as...,general_digital,general_digital
4,New photo from search,500px,Update device wallpaper,Android Device,New york,If new photo by anyone matching search new yor...,general_digital,general_digital


In [29]:
def rule_scope(row):
    """
    Assegna una categoria di rilevanza alla regola.

    Logica:
    - core_smart_home: trigger e action entrambi smart_home;
    - context_aware_smart_home: combinazione tra context e smart_home;
    - smart_home_related: almeno un lato è smart_home;
    - review: presenza di unknown;
    - out_of_scope: nessun segnale utile per il progetto.
    """
    t = row["trigger_class"]
    a = row["action_class"]

    if t == "smart_home" and a == "smart_home":
        return "core_smart_home"

    if (t == "context" and a == "smart_home") or (t == "smart_home" and a == "context"):
        return "context_aware_smart_home"

    if t == "smart_home" or a == "smart_home":
        return "smart_home_related"

    if t == "unknown" or a == "unknown":
        return "review"

    return "out_of_scope"

In [31]:
# Applicazione dello scope alle regole
df["rule_scope"] = df.apply(rule_scope, axis=1)

print("Distribuzione rule_scope:")
print(df["rule_scope"].value_counts(dropna=False).to_string())

Distribuzione rule_scope:
rule_scope
out_of_scope                69966
smart_home_related           3994
context_aware_smart_home     1804
review                       1797
core_smart_home              1653


In [32]:
# Estrazione dei subset utili al progetto
candidate_df = df[
    df["rule_scope"].isin([
        "core_smart_home",
        "context_aware_smart_home",
        "smart_home_related"
    ])
].copy()

review_df = df[df["rule_scope"] == "review"].copy()

print("Candidate shape:", candidate_df.shape)
print("Review shape:", review_df.shape)

Candidate shape: (7451, 9)
Review shape: (1797, 9)


In [33]:
# Esportazione dei file finali
candidate_df.to_csv(CANDIDATES_PATH, sep=";", index=False, encoding="utf-8-sig")
review_df.to_csv(REVIEW_PATH, sep=";", index=False, encoding="utf-8-sig")

print(f"Salvato candidate dataset in: {CANDIDATES_PATH}")
print(f"Salvato review dataset in: {REVIEW_PATH}")

Salvato candidate dataset in: ..\data\dataset_candidates_rag.csv
Salvato review dataset in: ..\data\dataset_review_rag.csv


In [34]:
# Sanity checks finali
print("Esempi candidate:")
display(candidate_df.head(10))

print("\nEsempi review:")
display(review_df.head(10))

Esempi candidate:


,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc,trigger_class,action_class,rule_scope
26,Item added to your Shopping List,Amazon Alexa,Send an SMS,Android SMS,"If item added to your To Do List, then send me...","If item added to your To Do List, then send me...",smart_home,general_digital,smart_home_related
27,Item added to your Shopping List,Amazon Alexa,Append to a text file,Dropbox,"Add new items to your Alexa Shopping List, app...",If a new item is added to your Alexa Shopping ...,smart_home,general_digital,smart_home_related
28,Item deleted on your Shopping List,Amazon Alexa,Send an email,Gmail,If shopping list item deleted send email,"If I delete an item in Alexa App, send me an u...",smart_home,general_digital,smart_home_related
29,Your Timer goes off,Amazon Alexa,Send a notification from the IFTTT app,Notifications,"If Timer Goes Off, A IF Notification Is Sent","When your Echo's timer goes off, a notificatio...",smart_home,general_digital,smart_home_related
30,Your Alarm goes off,Amazon Alexa,Send a notification from the IFTTT app,Notifications,Receive Notification on Alarm,"If Alexa alarm goes off, then send a notificat...",smart_home,general_digital,smart_home_related
31,New song played,Amazon Alexa,Send me an SMS,SMS,"If a song is played, then send me an SMS#newpl...","For bundled voice-activation Applets, includin...",smart_home,general_digital,smart_home_related
32,Item added to your To Do List,Amazon Alexa,Send me an SMS,SMS,Receive an SMS when an item is added to the Al...,Alexa To Do lists are shared among all users r...,smart_home,general_digital,smart_home_related
33,Item added to your Shopping List,Amazon Alexa,Send me an SMS,SMS,When an item is added to my shopping list send...,When an item is added to my shopping list send...,smart_home,general_digital,smart_home_related
203,Any phone call answered,Android Phone Call,End activity,Harmony,End a Harmony activity when you answer a call,"If any phone call answered, then end Harmony a...",general_digital,smart_home,smart_home_related
204,Any phone call missed,Android Phone Call,Run a HomeSeer system event,HomeSeer,"If any phone call missed, then run a HomeSeer ...",This recipe triggers when I miss a cell phone ...,general_digital,smart_home,smart_home_related



Esempi review:


,triggerTitle,triggerChannelTitle,actionTitle,actionChannelTitle,title,desc,trigger_class,action_class,rule_scope
144,Any phone call missed,Android Phone Call,Send notification,Comcast Labs,Missed phone call notification on Xfinity set ...,Missed phone call notification on Xfinity set ...,general_digital,unknown,review
205,Any phone call missed,Android Phone Call,Make a web request,Maker Webhooks,Log missed phone call in Zoho,"If phone call is missed, then try to find in Z...",general_digital,unknown,review
206,Any phone call answered,Android Phone Call,Make a web request,Maker Webhooks,Log answered phone call in Zoho,"If phone call is answered, then try to find in...",general_digital,unknown,review
411,Detects Motion,Arlo QA (Staging),Send an email,Gmail,"if Arlo detected motion, then send me email","if Arlo detected motion, then send me email",unknown,general_digital,review
412,Detects Motion,Arlo QA (Staging),Send me an SMS,SMS,"If Arlo Detects Motion, then send me an SMS","If Arlo Detects Motion, then send me an SMS",unknown,general_digital,review
413,Detects Motion,Arlo QA (Staging),Send me an SMS,SMS,"If Arlo Detects Motion, then send me an SMS","If Arlo Detects Motion, then send me an SMS",unknown,general_digital,review
414,Detects Motion,Arlo QA (Staging),Send me an SMS,SMS,"If Arlo Detects Motion, then send me an SMS","If Arlo Detects Motion, then send me an SMS",unknown,general_digital,review
417,Motion detected,Beseye_dev,Add reminder to list,iOS Reminders,Set a reminder when motion is detected,If you hate to receive every notification but ...,unknown,general_digital,review
818,You are tagged in a photo,Facebook,Send notification,Comcast Labs,"If Tagged in FB Photo, then send notification ...","If you are tagged in a FB photo, then you can ...",general_digital,unknown,review
899,You are tagged in a photo,Facebook,Send a notification,Google Glass,Facebook Tag to Google Glass,"When you get tagged in a photo on Facebook, yo...",general_digital,unknown,review
